
# R1 + R2: Determinism Check and Controlled n=8 vs n=20 Heterogeneity Replication

**Purpose.** Experiment 33 (B2a) found that the sensor-heterogeneity effect
from Experiments 25–26 did not replicate at `n_windows=20`: Panda MAE
increased 83% from `homo_matched` to `hetero_controlled` at `n=8`
(0.331 → 0.605) versus only 4% at `n=20` (0.5412 → 0.5636), on the
*same* hardcoded channel indices. This notebook runs the two checks
queued in the log's Replication Lane (R1, R2) before that finding is
cited at any confidence level in either direction.

**R1 — Determinism check.** Confirms `panda_forecast` is deterministic
(no unaccounted stochasticity) by running the identical window twice.

**R2 — Controlled n=8 vs n=20 replication.** Since window start positions
are computed as `np.linspace(0, max_start, n_windows, dtype=int)`
(confirmed from your `build_periodic_windows_CT` helper), the n=8 and
n=20 window sets are **not nested** — they are two different samplings
of the same span, sharing only the first and last start index. This
notebook recomputes both window sets exactly, evaluates both channel
subsets on both window sets in the same session (eliminating
implementation drift as a variable), and reports:
1. Whether the original n=8 aggregate MAE reproduces (rules out drift).
2. Per-window MAE and calendar date (day-of-year) for every window in
   both sets, to check whether the original n=8 windows clustered in a
   particular season (competing explanation 2 in the log).

**Pre-registered decision rule** (fixed before running): if the
recomputed n=8 aggregate MAE for `homo_matched` and `hetero_controlled`
at H=96 reproduces the logged Experiment 25 values (0.331, 0.605) within
Monte-Carlo-negligible tolerance (<0.005 absolute, since this is a
deterministic recomputation, not a new random draw), implementation
drift is ruled out and the non-replication is attributed to sample
size / window-selection effects (explanations 1–2). If it does *not*
reproduce, implementation drift (explanation 3) is implicated and takes
priority to resolve before anything else.

**Cell 2 below** contains your verbatim setup (Cells 1-3 from
`new_experiments.ipynb`, i.e. imports/config, model loading, and all
helper functions including `evaluate()`), plus the `data_weather`
loading line pulled in from the Priority-1 cell where it originally
lived. Nothing in Cell 2 has been rewritten -- it is copied exactly,
which matters here specifically because implementation drift is one of
the three competing explanations for the non-replication, and a
reconstruction risks introducing the very confound this notebook exists
to rule out. If your local copy of `new_experiments.ipynb` has diverged
from this (e.g. you patched something after these cells were captured),
replace Cell 2 with your current version before running.


In [1]:

# =====================================================================
# Verbatim setup, reused from new_experiments.ipynb Cells 1-3, plus the
# data_weather loading line (originally in the Priority 1 cell, not in
# Cells 1-3 -- pulled in here so this notebook is self-contained).
# =====================================================================

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq
from scipy.linalg import svd
from scipy.stats import wilcoxon, linregress
from scipy.integrate import solve_ivp
from sklearn.metrics import pairwise_distances
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

N_WINDOWS   = 8
CONTEXT_LEN = 512
PRED_LEN    = 96
DATA_DIR    = './ts_data'  # adjust if needed

import sys
sys.path.insert(0, './panda')

from panda.patchtst.pipeline import PatchTSTPipeline
from chronos import ChronosPipeline

panda_model = PatchTSTPipeline.from_pretrained(
    mode='predict',
    pretrain_path='GilpinLab/panda',
    device_map=device,
)

chronos_model = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map=device,
    torch_dtype=torch.bfloat16,
)

print('Models loaded.')

# -------------------------------------------------------
# Metrics
# -------------------------------------------------------
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred)**2))

# -------------------------------------------------------
# Per-window normalisation
# -------------------------------------------------------
def instance_norm_window(x_CT):
    '''x_CT: (C, T). Normalise per channel using this window only.'''
    mu  = x_CT.mean(axis=1, keepdims=True)
    std = x_CT.std( axis=1, keepdims=True) + 1e-8
    return (x_CT - mu) / std, mu, std

def load_ts(path):
    '''Raw (C, T) -- no global normalisation.'''
    df = pd.read_csv(path)
    df = df.select_dtypes(include=[np.number])
    return df.values.astype(np.float32).T  # (C, T)

# -------------------------------------------------------
# Inference
# -------------------------------------------------------
def panda_forecast(context_np, horizon):
    '''context_np: (C, T) normalised. Returns (C, horizon).'''
    TRAIN_H   = 128
    remaining = horizon
    ctx       = context_np.copy()
    preds     = []
    while remaining > 0:
        h         = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = panda_model.predict(
                context_t, h,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
    return np.concatenate(preds, axis=1)  # (C, horizon)

def chronos_forecast(context_np, horizon):
    '''Batched -- all channels in one call.'''
    ctx = torch.tensor(context_np, dtype=torch.float32)
    with torch.no_grad():
        out = chronos_model.predict(
            ctx, prediction_length=horizon, num_samples=1
        )
    return out[:, 0, :].cpu().numpy()  # (C, horizon)

# -------------------------------------------------------
# Core evaluator (unchanged from new_experiments.ipynb)
# -------------------------------------------------------
def evaluate(data_CT, horizon, n_windows=N_WINDOWS, label='',
             fn_a=None, fn_b=None,
             name_a='panda', name_b='chronos'):
    '''
    data_CT: (C, T) RAW. Normalises each window independently.
    fn_a, fn_b: (context_normed: (C,T), horizon) -> (C, H)
    '''
    if fn_a is None: fn_a = panda_forecast
    if fn_b is None: fn_b = chronos_forecast

    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f'  [SKIP] {label}: T={T} too short')
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_a, mae_b = [], []

    for s in starts:
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std
        mae_a.append(mae(tgt_norm, fn_a(ctx_norm, horizon)))
        mae_b.append(mae(tgt_norm, fn_b(ctx_norm, horizon)))

    diff = np.array(mae_b) - np.array(mae_a)
    try:
        _, pval = wilcoxon(diff, alternative='greater') \
            if np.any(diff != 0) else (0, 1.0)
    except Exception:
        pval = np.nan

    adv = np.median(mae_b) - np.median(mae_a)
    sig = ' *' if pval < 0.05 else (' ~' if pval < 0.10 else '')
    iqr_a = np.percentile(mae_a,75) - np.percentile(mae_a,25)
    iqr_b = np.percentile(mae_b,75) - np.percentile(mae_b,25)

    result = {
        'label'         : label,
        'horizon'       : horizon,
        'name_a'        : name_a,
        'name_b'        : name_b,
        f'{name_a}_mae' : np.median(mae_a),
        f'{name_a}_iqr' : iqr_a,
        f'{name_b}_mae' : np.median(mae_b),
        f'{name_b}_iqr' : iqr_b,
        'advantage_mae' : adv,
        'wilcoxon_p'    : pval,
    }
    print(
        f'  {label:50s}  H={horizon:4d}  '
        f'{name_a}={np.median(mae_a):.4f}[+/-{iqr_a:.4f}]  '
        f'{name_b}={np.median(mae_b):.4f}[+/-{iqr_b:.4f}]  '
        f'Adv={adv:+.4f}  p={pval:.3f}{sig}'
    )
    return result

print('Helpers defined.')

# -------------------------------------------------------
# Data loading (originally in the Priority 1 cell)
# -------------------------------------------------------
data_weather = load_ts(f'{DATA_DIR}/weather.csv')
print(f'Weather shape: {data_weather.shape}')


Device: cpu
Models loaded.
Helpers defined.
Weather shape: (21, 52696)



## R1 — Determinism Check

Runs `panda_forecast` twice on the identical context window and confirms
the output is bit-for-bit (or numerically, within float tolerance)
identical. If Panda's forecast involves any sampling step (e.g.
quantile sampling rather than a deterministic point forecast), this
will show nonzero difference and needs to be accounted for before R2's
comparisons mean anything.


In [2]:

print('R1: Determinism check on panda_forecast')
print('-' * 70)

H_CHECK = 96
rng_check_start = 0  # first available window start

ctx_raw = data_weather[:, rng_check_start : rng_check_start + CONTEXT_LEN]
ctx_norm, mu, std = instance_norm_window(ctx_raw)

fc_1 = panda_forecast(ctx_norm, H_CHECK)
fc_2 = panda_forecast(ctx_norm, H_CHECK)

fc_1 = np.asarray(fc_1)
fc_2 = np.asarray(fc_2)

max_abs_diff = float(np.max(np.abs(fc_1 - fc_2)))
mean_abs_diff = float(np.mean(np.abs(fc_1 - fc_2)))
identical = np.array_equal(fc_1, fc_2)

print(f'Forecast shape: {fc_1.shape}')
print(f'Bit-for-bit identical: {identical}')
print(f'Max abs difference:    {max_abs_diff:.10f}')
print(f'Mean abs difference:   {mean_abs_diff:.10f}')

if identical or max_abs_diff < 1e-6:
    print('\nVERDICT: panda_forecast is deterministic (or deterministic to '
          'float precision). Any n=8/n=20 discrepancy is not attributable '
          'to per-call stochasticity.')
else:
    print('\nVERDICT: panda_forecast is NOT deterministic. This must be '
          'resolved before R2 results can be interpreted -- repeated calls '
          'on identical input are producing different output, which means '
          'single-run MAE values (at any n) carry unaccounted variance.')

r1_result = {
    'identical': identical,
    'max_abs_diff': max_abs_diff,
    'mean_abs_diff': mean_abs_diff,
}


R1: Determinism check on panda_forecast
----------------------------------------------------------------------
Forecast shape: (21, 96)
Bit-for-bit identical: True
Max abs difference:    0.0000000000
Mean abs difference:   0.0000000000

VERDICT: panda_forecast is deterministic (or deterministic to float precision). Any n=8/n=20 discrepancy is not attributable to per-call stochasticity.



## Channel Subset Definitions

Hardcoded from the confirmed Experiment 25/26 notebook output (channel
clustering cell). These are the exact indices used in the original
n=8 runs.


In [3]:

# Confirmed from Experiment 24/25/26 notebook output.
homo_matched       = np.array([6, 8, 5, 3, 7, 2, 1])
hetero_controlled  = np.array([16, 4, 19, 7, 12, 2, 20])

SUBSETS = {
    'homo_matched': homo_matched,
    'hetero_controlled': hetero_controlled,
}

print('Subsets:')
for name, idx in SUBSETS.items():
    print(f'  {name:>20}: {idx.tolist()}')


Subsets:
          homo_matched: [6, 8, 5, 3, 7, 2, 1]
     hetero_controlled: [16, 4, 19, 7, 12, 2, 20]



## R2 Core — `evaluate_at_starts`

A generalisation of your `evaluate()` that takes an explicit array of
window start positions rather than computing its own via `n_windows`.
This lets us force both the n=8 and n=20 window sets to be evaluated
on demand, in the same session, for both subsets, with per-window MAE
retained (not just the aggregate) so we can inspect individual windows
and their calendar dates.


In [4]:

def evaluate_at_starts(data_CT, horizon, starts, label=''):
    '''
    Same evaluation logic as evaluate() above, but takes explicit window
    start positions instead of computing them internally via n_windows.
    Matches evaluate() exactly: MAE is computed in normalised space
    (mae(tgt_norm, forecast(ctx_norm, horizon))), and aggregation uses
    the median, not the mean, per your original convention. Returns
    per-window MAE arrays (not just the aggregate) plus the start
    indices, so R2 can compare window-for-window across conditions.
    '''
    panda_maes   = []
    chronos_maes = []

    for s in starts:
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm           = (tgt_raw - mu) / std

        panda_fc_norm   = panda_forecast(ctx_norm, horizon)
        chronos_fc_norm = chronos_forecast(ctx_norm, horizon)

        panda_maes.append(mae(tgt_norm, panda_fc_norm))
        chronos_maes.append(mae(tgt_norm, chronos_fc_norm))

    panda_maes   = np.array(panda_maes)
    chronos_maes = np.array(chronos_maes)

    diff = chronos_maes - panda_maes
    try:
        _, p = wilcoxon(diff, alternative='greater') \
            if np.any(diff != 0) else (0, 1.0)
    except Exception:
        p = float('nan')

    result = {
        'label': label,
        'horizon': horizon,
        'n_windows': len(starts),
        'starts': list(starts),
        'panda_mae': float(np.median(panda_maes)),
        'panda_iqr': float(np.percentile(panda_maes, 75) - np.percentile(panda_maes, 25)),
        'chronos_mae': float(np.median(chronos_maes)),
        'chronos_iqr': float(np.percentile(chronos_maes, 75) - np.percentile(chronos_maes, 25)),
        'advantage_mae': float(np.median(chronos_maes) - np.median(panda_maes)),
        'wilcoxon_p': float(p),
        'panda_maes_per_window': panda_maes,
        'chronos_maes_per_window': chronos_maes,
    }
    return result

print('evaluate_at_starts defined.')


evaluate_at_starts defined.



## Window Start Positions and Seasonal Diagnostic

Recomputes `starts_n8` and `starts_n20` exactly as your harness does
(`np.linspace(0, max_start, n_windows, dtype=int)`), for both H=96 and
H=336, and converts each start index to day-of-year (Weather is
10-minute resolution, 144 samples/day) to check whether the original
n=8 windows clustered in a particular season relative to the n=20 set.


In [5]:

SAMPLES_PER_DAY = 144  # Weather: 10-minute resolution

def starts_for(n_windows, horizon, data_CT=data_weather):
    C, T = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    return np.linspace(0, max_start, n_windows, dtype=int)

def day_of_year(start_idx):
    return (start_idx / SAMPLES_PER_DAY) % 365.25

HORIZONS = [96, 336]
starts_by_h = {}

for h in HORIZONS:
    s8  = starts_for(8, h)
    s20 = starts_for(20, h)
    starts_by_h[h] = {'n8': s8, 'n20': s20}

    print(f'H={h}')
    print(f'  n=8  starts: {s8.tolist()}')
    print(f'       days:   {[round(day_of_year(s), 1) for s in s8]}')
    print(f'  n=20 starts: {s20.tolist()}')
    print(f'       days:   {[round(day_of_year(s), 1) for s in s20]}')
    print(f'  n=8 subset of n=20? {set(s8.tolist()).issubset(set(s20.tolist()))}')
    print()


H=96
  n=8  starts: [0, 7441, 14882, 22323, 29764, 37205, 44646, 52088]
       days:   [0.0, 51.7, 103.3, 155.0, 206.7, 258.4, 310.0, 361.7]
  n=20 starts: [0, 2741, 5482, 8224, 10965, 13707, 16448, 19190, 21931, 24673, 27414, 30156, 32897, 35639, 38380, 41122, 43863, 46605, 49346, 52088]
       days:   [0.0, 19.0, 38.1, 57.1, 76.1, 95.2, 114.2, 133.3, 152.3, 171.3, 190.4, 209.4, 228.5, 247.5, 266.5, 285.6, 304.6, 323.6, 342.7, 361.7]
  n=8 subset of n=20? False

H=336
  n=8  starts: [0, 7406, 14813, 22220, 29627, 37034, 44441, 51848]
       days:   [0.0, 51.4, 102.9, 154.3, 205.7, 257.2, 308.6, 360.1]
  n=20 starts: [0, 2728, 5457, 8186, 10915, 13644, 16373, 19101, 21830, 24559, 27288, 30017, 32746, 35474, 38203, 40932, 43661, 46390, 49119, 51848]
       days:   [0.0, 18.9, 37.9, 56.8, 75.8, 94.8, 113.7, 132.6, 151.6, 170.5, 189.5, 208.5, 227.4, 246.3, 265.3, 284.2, 303.2, 322.2, 341.1, 360.1]
  n=8 subset of n=20? False




## R2 Main Run

Evaluates both subsets (`homo_matched`, `hetero_controlled`) at both
window sets (n=8, n=20), at H=96 and H=336, in this single session.


In [6]:

r2_results = []

for h in HORIZONS:
    for wname, starts in starts_by_h[h].items():
        for sname, idx in SUBSETS.items():
            data_sub = data_weather[idx, :]
            label = f'{sname}_{wname}_H{h}'
            r = evaluate_at_starts(data_sub, h, starts, label=label)
            r['subset'] = sname
            r['window_set'] = wname
            r2_results.append(r)
            print(f'{label:>35}  panda={r["panda_mae"]:.4f}  '
                  f'chronos={r["chronos_mae"]:.4f}  '
                  f'adv={r["advantage_mae"]:+.4f}  p={r["wilcoxon_p"]:.4f}')

print('\nDone.')


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

                homo_matched_n8_H96  panda=0.3309  chronos=0.7024  adv=+0.3715  p=0.0078


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

           hetero_controlled_n8_H96  panda=0.6126  chronos=0.8561  adv=+0.2435  p=0.0078


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

...

          hetero_controlled_n8_H336  panda=0.8669  chronos=1.0580  adv=+0.1911  p=0.0273


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

              homo_matched_n20_H336  panda=0.9220  chronos=1.1806  adv=+0.2586  p=0.0002


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

         hetero_controlled_n20_H336  panda=0.7084  chronos=1.0852  adv=+0.3768  p=0.0000

Done.



## Check 1 — Reproducibility Against Logged Experiment 25/26 Values

If the recomputed n=8 aggregate MAE matches the logged values at both
horizons (0.331/0.605 at H=96, 0.841/0.867 at H=336, for
`homo_matched`/`hetero_controlled` respectively) within a small
tolerance, implementation drift is ruled out as an explanation for the
non-replication, per the pre-registered decision rule above. Covering
both horizons also directly tests whether `panda_forecast`'s
autoregressive rollout (used for H=336, chained 128-step calls) behaves
identically through `evaluate_at_starts` as it does through the
original `evaluate()` -- if H=96 matches but H=336 doesn't, that
isolates any drift to the rollout path specifically.


In [7]:

LOGGED_EXP25_26 = {
    ('homo_matched', 96):        0.331,   # Experiment 25
    ('hetero_controlled', 96):   0.605,   # Experiment 26
    ('homo_matched', 336):       0.841,   # Experiment 25
    ('hetero_controlled', 336):  0.867,   # Experiment 26
}
TOLERANCE = 0.005

print('Reproducibility check (n=8, against logged Experiment 25/26 values)')
print('-' * 70)

drift_flag = False
for (sname, h), logged_mae in LOGGED_EXP25_26.items():
    match = [r for r in r2_results
             if r['subset'] == sname and r['window_set'] == 'n8' and r['horizon'] == h][0]
    recomputed = match['panda_mae']
    diff = abs(recomputed - logged_mae)
    status = 'MATCH' if diff < TOLERANCE else 'MISMATCH'
    if diff >= TOLERANCE:
        drift_flag = True
    print(f'  {sname:>20}  H={h:>3}: logged={logged_mae:.4f}  recomputed={recomputed:.4f}  '
          f'diff={diff:.4f}  [{status}]')

print()
if drift_flag:
    print('VERDICT: at least one subset/horizon does not reproduce the logged '
          'n=8 value within tolerance. Implementation drift (competing '
          'explanation 3) is implicated and should be resolved before the '
          'n=8/n=20 comparison below is interpreted. If specifically the '
          'H=336 rows mismatch while H=96 rows match, that isolates the '
          'issue to the autoregressive rollout path in panda_forecast '
          'rather than the evaluation loop in general.')
else:
    print('VERDICT: all four subset/horizon combinations reproduce the '
          'logged n=8 values, including H=336. This confirms the '
          'autoregressive rollout in panda_forecast behaves identically '
          'whether called through evaluate() or evaluate_at_starts(), and '
          'rules out implementation drift entirely (not just at H=96). '
          'The n=8/n=20 discrepancy is attributable to sample-size and/or '
          'window-selection effects (competing explanations 1-2).')


Reproducibility check (n=8, against logged Experiment 25/26 values)
----------------------------------------------------------------------
          homo_matched  H= 96: logged=0.3310  recomputed=0.3309  diff=0.0001  [MATCH]
     hetero_controlled  H= 96: logged=0.6050  recomputed=0.6126  diff=0.0076  [MISMATCH]
          homo_matched  H=336: logged=0.8410  recomputed=0.8331  diff=0.0079  [MISMATCH]
     hetero_controlled  H=336: logged=0.8670  recomputed=0.8669  diff=0.0001  [MATCH]

VERDICT: at least one subset/horizon does not reproduce the logged n=8 value within tolerance. Implementation drift (competing explanation 3) is implicated and should be resolved before the n=8/n=20 comparison below is interpreted. If specifically the H=336 rows mismatch while H=96 rows match, that isolates the issue to the autoregressive rollout path in panda_forecast rather than the evaluation loop in general.



## Check 2 — n=8 vs n=20 Comparison and Seasonal Spread

Direct comparison of the advantage at n=8 vs n=20 for both subsets
(this is expected to reproduce the Experiment 33 pattern, now computed
in the same run as the reproducibility check above), plus the
day-of-year spread of the n=8 windows relative to the n=20 windows, to
test the seasonal-clustering explanation directly.


In [8]:

import pandas as pd

df_r2 = pd.DataFrame([
    {k: v for k, v in r.items()
     if k not in ('panda_maes_per_window', 'chronos_maes_per_window', 'starts')}
    for r in r2_results
])

print('=== n=8 vs n=20 Advantage Comparison ===')
print(df_r2[['subset', 'window_set', 'horizon', 'panda_mae', 'chronos_mae',
             'advantage_mae', 'wilcoxon_p']].to_string(index=False))

print()
print('=== Seasonal Spread Check (H=96) ===')
for wname in ['n8', 'n20']:
    s = starts_by_h[96][wname]
    days = np.array([day_of_year(x) for x in s])
    print(f'  {wname}: days={np.round(days, 1).tolist()}  '
          f'span={days.max()-days.min():.1f} days  std={days.std():.1f}')

s8_days  = np.array([day_of_year(x) for x in starts_by_h[96]['n8']])
s20_days = np.array([day_of_year(x) for x in starts_by_h[96]['n20']])
print(f'\n  n=8 day-of-year std:  {s8_days.std():.1f}')
print(f'  n=20 day-of-year std: {s20_days.std():.1f}')
if s8_days.std() < 0.6 * s20_days.std():
    print('  -> n=8 windows are notably more clustered in time than n=20.')
    print('     Seasonal-clustering explanation (competing explanation 2) is supported.')
else:
    print('  -> n=8 windows are not markedly more clustered than n=20.')
    print('     Seasonal-clustering explanation is not strongly supported by this check;')
    print('     small-sample noise (competing explanation 1) becomes the leading account.')


=== n=8 vs n=20 Advantage Comparison ===
           subset window_set  horizon  panda_mae  chronos_mae  advantage_mae  wilcoxon_p
     homo_matched         n8       96   0.330867     0.702415       0.371548    0.007812
hetero_controlled         n8       96   0.612616     0.856069       0.243454    0.007812
     homo_matched        n20       96   0.553930     0.817299       0.263369    0.006040
hetero_controlled        n20       96   0.560874     0.924654       0.363780    0.000131
     homo_matched         n8      336   0.833083     1.178727       0.345644    0.003906
hetero_controlled         n8      336   0.866903     1.057965       0.191062    0.027344
     homo_matched        n20      336   0.922005     1.180557       0.258552    0.000161
hetero_controlled        n20      336   0.708394     1.085175       0.376781    0.000002

=== Seasonal Spread Check (H=96) ===
  n8: days=[0.0, 51.7, 103.3, 155.0, 206.7, 258.4, 310.0, 361.7]  span=361.7 days  std=118.4
  n20: days=[0.0, 19.0, 38.


## Save Results


In [9]:

df_r2.to_csv('r1_r2_results.csv', index=False)

with open('r1_r2_summary.txt', 'w') as f:
    f.write('R1 determinism check:\n')
    f.write(f'  identical={r1_result["identical"]}  '
            f'max_abs_diff={r1_result["max_abs_diff"]:.10f}\n\n')
    f.write('R2 n=8 vs n=20 comparison:\n')
    f.write(df_r2[['subset', 'window_set', 'horizon', 'panda_mae',
                    'chronos_mae', 'advantage_mae', 'wilcoxon_p']].to_string(index=False))
    f.write('\n')

print('Saved r1_r2_results.csv and r1_r2_summary.txt')
print('\nPaste back: the R1 verdict line, the reproducibility check block, '
      'the n=8-vs-n=20 table, and the seasonal spread block.')


Saved r1_r2_results.csv and r1_r2_summary.txt

Paste back: the R1 verdict line, the reproducibility check block, the n=8-vs-n=20 table, and the seasonal spread block.
